In [19]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.Routes import RUTAS
from pandas.api.types import is_float_dtype, is_integer_dtype
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [20]:
# Cargar dataset original
df = pd.read_excel(RUTAS['data_raw'] / 'Dirty_Sales_Marketing_Dataset.xlsx')
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")


Dataset cargado: 15750 filas x 30 columnas


In [21]:
# Optimización de memoria con downcasting para demostrar impacto cuantitativo
memoria_antes = df.memory_usage(deep=True).sum()

# Se aplica downcasting en columnas numéricas para reducir huella de memoria sin perder semántica.
columnas_numericas_optimizar = [
    'avg_order_value', 'lifetime_value', 'last_3_month_purchase_freq'
    ,'marketing_spend_per_user', 'avg_session_time', 'pages_per_session'
    ,'email_open_rate', 'email_click_rate', 'satisfaction_score', 'nps_score'
    ,'total_visits', 'total_spent'
]
for columna in columnas_numericas_optimizar:
    if columna in df.columns:
        if is_float_dtype(df[columna]):
            df[columna] = pd.to_numeric(df[columna], errors='coerce', downcast='float')
        elif is_integer_dtype(df[columna]):
            df[columna] = pd.to_numeric(df[columna], errors='coerce', downcast='integer')

memoria_despues = df.memory_usage(deep=True).sum()
memoria_ahorrada = memoria_antes - memoria_despues
porcentaje_ahorro = (memoria_ahorrada / memoria_antes) * 100 if memoria_antes else 0

print(f"Memoria antes: {memoria_antes:,} bytes")
print(f"Memoria después: {memoria_despues:,} bytes")
print(f"Ahorro: {memoria_ahorrada:,} bytes ({porcentaje_ahorro:.2f}%)")

Memoria antes: 11,029,469 bytes
Memoria después: 10,131,719 bytes
Ahorro: 897,750 bytes (8.14%)


1. Transformación y normalización de columnas numéricas


In [22]:
columnas_numericas = [
    'age', 'total_spent', 'avg_order_value', 'lifetime_value',
    'last_3_month_purchase_freq', 'marketing_spend_per_user',
    'total_visits', 'avg_session_time', 'pages_per_session',
    'email_open_rate', 'email_click_rate', 'support_tickets',
    'delivery_delay_days', 'satisfaction_score', 'nps_score'
]


In [23]:
# age: limpiar espacios, reemplazar 'nan'/'NAN' como texto -> float
df['age'] = df['age'].astype(str).str.strip()
df['age'] = df['age'].replace({'nan': np.nan, 'NAN': np.nan, 'NaN': np.nan, '': np.nan})
df['age'] = pd.to_numeric(df['age'], errors='coerce')


In [24]:
# age se inputan los valores null con mediana global
age_mediana = df['age'].median()
df['age'] = df['age'].fillna(age_mediana).astype(int)
print(f"age → mediana usada: {age_mediana}")

# total_spent se inputa con mediana por subscription_type
df['total_spent'] = df.groupby('subscription_type')['total_spent'].transform(
    lambda x: x.fillna(x.median())
)
df['total_spent'] = df['total_spent'].fillna(df['total_spent'].median())
print(f"total_spent → mediana utilizada: {df['total_spent'].median():.2f}")

# satisfaction_score se inputan los valores null con mediana global
sat_mediana = df['satisfaction_score'].median()
df['satisfaction_score'] = df['satisfaction_score'].fillna(sat_mediana)
print(f"satisfaction_score → mediana usada: {sat_mediana}")


age → mediana usada: 35.0
total_spent → mediana utilizada: 499.01
satisfaction_score → mediana usada: 4.0


### 1.1 Normalización de outliers

Para las variables numéricas con valores extremos se aplica winsorización basada en IQR. Esto conserva todas las filas del dataset y limita los valores fuera de los umbrales $Q1 - 1.5 \times IQR$ y $Q3 + 1.5 \times IQR$.

Se aplica a variables continuas y de conteo donde los outliers pueden distorsionar el análisis: `age`, `total_spent`, `avg_order_value`, `lifetime_value`, `total_visits`, `avg_session_time`, `pages_per_session`, `support_tickets` y `delivery_delay_days`.

In [25]:
columnas_outliers = [
    'age', 'total_spent', 'avg_order_value', 'lifetime_value',
    'total_visits', 'avg_session_time', 'pages_per_session',
    'support_tickets', 'delivery_delay_days'
]

resumen_outliers = []
for columna in columnas_outliers:
    serie = pd.to_numeric(df[columna], errors='coerce')
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers_antes = int(((serie < limite_inferior) | (serie > limite_superior)).sum())
    df[columna] = serie.clip(lower=limite_inferior, upper=limite_superior)
    serie_despues = pd.to_numeric(df[columna], errors='coerce')
    outliers_despues = int(((serie_despues < limite_inferior) | (serie_despues > limite_superior)).sum())

    resumen_outliers.append({
        'columna': columna,
        'lim_inf': round(limite_inferior, 3),
        'lim_sup': round(limite_superior, 3),
        'outliers_antes': outliers_antes,
        'outliers_despues': outliers_despues
    })

# Restaurar tipo entero en columnas discretas
for columna in ['age', 'total_visits', 'support_tickets', 'delivery_delay_days']:
    df[columna] = df[columna].round().astype(int)

pd.DataFrame(resumen_outliers)

,columna,lim_inf,lim_sup,outliers_antes,outliers_despues
0,age,13.500,57.500,698,0
1,total_spent,-232.297,1231.482,126,0
2,avg_order_value,-7.747,127.634,57,0
3,lifetime_value,-665.865,3082.785,58,0
4,total_visits,3.000,27.000,42,0
5,avg_session_time,-0.129,16.173,46,0
6,pages_per_session,-0.047,8.046,39,0
7,support_tickets,-2.000,6.000,85,0
8,delivery_delay_days,-1.000,7.000,195,0


In [26]:
# Finalmente se redondean las columnas numericas a 3 decimales excepto age, support_tickets y delivery_delay_days
columnas_a_redondear = [col for col in columnas_numericas if col not in ['age', 'support_tickets', 'delivery_delay_days']]
df[columnas_a_redondear] = df[columnas_a_redondear].round(3)

## 2. Transformación y normalización de Columnas categóricas seleccionadas

In [27]:
columnas_categoricas = ['gender', 'country', 'acquisition_channel', 'subscription_type', 'payment_method']


In [28]:
# gender: strip + title + unificar variantes 'nan' -> 'unknown'
df['gender'] = df['gender'].astype(str).str.strip().str.title()
df['gender'] = df['gender'].replace({'Nan': np.nan, 'None': np.nan, '': np.nan})
df['gender'] = df['gender'].fillna('Unknown')


In [29]:
# Para country se imputan los valores null con la moda global (la mayoría de los países son USA, por eso se elige esta estrategia)
country_moda = df['country'].mode()[0]
df['country'] = df['country'].fillna(country_moda)


In [30]:
# Se aplica solo estandarización de formato (strip) para las columans que no tienen null.

for col in ['acquisition_channel', 'subscription_type', 'payment_method']:
    df[col] = df[col].astype(str).str.strip().str.title()


In [31]:

# Se consolida el subconjunto de 20 columnas (15 numéricas + 5 categóricas) con todas las transformaciones aplicadas y se exporta a `data/processed/`

cols_finales = columnas_numericas + columnas_categoricas
df_limpio = df[cols_finales].copy()

In [32]:
# Agrupación multivariable para analizar métricas clave por país y tipo de suscripción
agregacion_pais_suscripcion = df_limpio.groupby(['country', 'subscription_type']).agg(
    total_spent_mean=('total_spent', 'mean'),
    total_spent_max=('total_spent', 'max'),
    customer_count=('age', 'size')
)

agregacion_pais_suscripcion

total_spent_mean  total_spent_max  \
country    subscription_type                                      
Bangladesh Annual                   516.776501         1231.482   
           Monthly                  513.744468         1231.482   
Germany    Annual                   514.298807         1231.482   
           Monthly                  507.452867         1231.482   
India      Annual                   504.431058         1231.482   
           Monthly                  508.260050         1231.482   
UK         Annual                   510.455329         1231.482   
           Monthly                  498.427498         1231.482   
USA        Annual                   516.404051         1231.482   
           Monthly                  501.340928         1231.482   

                              customer_count  
country    subscription_type                  
Bangladesh Annual                       1369  
           Monthly                      1464  
Germany    Annual                       2185  
           Monthly                      2283  
India      Annual                       1385  
           Monthly                      1437  
UK         Annual                       1360  
           Monthly                      1451  
USA        Annual                       1422  
           Monthly                      1394

In [33]:
# Tabla dinámica para analizar adquisición por país.
tabla_dinamica_adquisicion_pais = pd.pivot_table(
    df_limpio[['acquisition_channel', 'country', 'total_spent']],
    index='country',
    columns='acquisition_channel',
    values='total_spent',
    aggfunc=['mean', 'count'],
    fill_value=0
)

tabla_dinamica_adquisicion_pais

mean                                       \
acquisition_channel       Email Facebook Ads  Google Ads     Organic   
country                                                                
Bangladesh           518.512041   514.047083  514.349337  516.452289   
Germany              488.816354   515.457972  518.051336  516.438584   
India                504.686793   507.176260  506.430233  511.684943   
UK                   517.928977   499.184092  500.968828  503.949458   
USA                  520.493013   518.916722  496.081738  516.480089   

                                count                                           
acquisition_channel    Referral Email Facebook Ads Google Ads Organic Referral  
country                                                                         
Bangladesh           512.800456   556          576        570     561      570  
Germany              515.779451   913          888        901     927      839  
India                501.390537   518          562        597     594      551  
UK                   499.931285   533          554        564     589      571  
USA                  492.908811   556          590        553     540      577

In [34]:
# Se codifican las variables categóricas con Label Encoding para facilitar su uso en modelos de machine learning.
from sklearn.preprocessing import LabelEncoder
df_codificado = df_limpio.copy()
labEn = LabelEncoder()
for col in columnas_categoricas:
    df_codificado[col] = labEn.fit_transform(df_codificado[col])
df_codificado.head()

,age,total_spent,avg_order_value,lifetime_value,last_3_month_purchase_freq,marketing_spend_per_user,total_visits,avg_session_time,pages_per_session,email_open_rate,email_click_rate,support_tickets,delivery_delay_days,satisfaction_score,nps_score,gender,country,acquisition_channel,subscription_type,payment_method
0,35,559.525,65.247,915.311,14,27.559999,7,13.904,5.415,0.67,0.26,0,3,3.0,10,1,2,0,0,4
1,35,356.491,48.474,2079.961,11,15.150000,19,5.113,5.352,0.70,0.37,5,3,3.0,7,3,1,3,1,0
2,27,689.332,77.815,1379.151,9,13.510000,18,9.743,3.595,0.47,0.44,1,2,5.0,6,0,1,0,0,4
3,36,445.430,71.712,774.653,7,25.650000,16,9.643,2.950,0.58,0.37,0,2,4.0,6,0,2,1,0,2
4,29,686.286,44.990,87.680,11,12.390000,12,7.791,2.406,0.05,0.16,2,4,3.0,1,1,4,4,1,0


In [35]:
# Finalmente se guarda el dataset limpio y codificado en formato Excel y CSV para su uso posterior en análisis o modelado.

archivo_resultante_excel = RUTAS['data_processed'] / 'Sales_Marketing_Clean.xlsx'
archivo_resultante_csv = RUTAS['data_processed'] / 'Sales_Marketing_Clean_(Codificado).csv'
df_limpio.to_excel(archivo_resultante_excel, index=False)
df_codificado.to_csv(archivo_resultante_csv, index=False)
print(f'Datasets guardados en: {archivo_resultante_excel} y \n{archivo_resultante_csv}')

Datasets guardados en: c:\Users\encar\Desktop\Sales_Marketing_Dataset\data\processed\Sales_Marketing_Clean.xlsx y 
c:\Users\encar\Desktop\Sales_Marketing_Dataset\data\processed\Sales_Marketing_Clean_(Codificado).csv
